# IRIS (Interface Region Imaging Spectrograph) — Key Algorithms Implementation
# IRIS (인터페이스 영역 영상 분광기) — 핵심 알고리즘 구현

**Paper**: De Pontieu, B. et al. (2014), *Solar Physics*, 289, 2733–2779

This notebook implements key algorithms and concepts from the IRIS instrument paper:
1. **CCD Dark Current Model** — Equation 1 from Section 7.1
2. **IRIS Thermal Coverage Visualization** — Temperature coverage of spectral lines (Table 4)
3. **Raster Scan Simulation** — Demonstrating dense, sparse, and sit-and-stare modes (Section 5)
4. **Rice Compression** — Lossless compression algorithm (Section 7.4)
5. **Orbital Wobble Model** — Thermal flexing correction (Section 7.5)

이 노트북은 IRIS 기기 논문의 핵심 알고리즘과 개념을 구현합니다:
1. **CCD 다크 전류 모델** — Section 7.1의 수식 1
2. **IRIS 열적 커버리지 시각화** — 분광선의 온도 범위 (Table 4)
3. **래스터 스캔 시뮬레이션** — 밀집, 성긴, sit-and-stare 모드 (Section 5)
4. **Rice 압축** — 무손실 압축 알고리즘 (Section 7.4)
5. **궤도 워블 모델** — 열변형 보정 (Section 7.5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. CCD Dark Current Model / CCD 다크 전류 모델

From Equation 1 (Section 7.1, p. 2758), the total dark level in read port $j$ is:

$$D_j = P_j[T_{\text{CEB}_j}(t - \delta t_j)] + e^{(a_j + b_j T_{\text{CCD}_j})} n_x n_y t_{\text{int}} + \Delta D_j(x, n_x, n_y, t_{\text{int}})$$

We model each component:
- **Pedestal**: constant offset depending on CEB temperature
- **Dark current**: exponentially dependent on CCD temperature, proportional to summing × integration time
- **Shape correction**: wavelength-dependent variation (simplified as linear ramp)

Section 7.1의 수식 1(p. 2758)에서, 읽기 포트 $j$의 총 다크 레벨은 위와 같습니다.
각 성분을 모델링합니다:
- **페데스탈**: CEB 온도에 의존하는 상수 오프셋
- **다크 전류**: CCD 온도에 지수적으로 의존, 합산 × 적분 시간에 비례
- **형상 보정**: 파장 방향 변화 (선형 램프로 단순화)

In [ ]:
def iris_dark_model(
    t_ccd: float,
    t_ceb: float,
    t_int: float,
    nx: int = 1,
    ny: int = 1,
    npix_x: int = 2061,
    a: float = -5.0,
    b: float = 0.15,
    pedestal_base: float = 85.0,
    pedestal_slope: float = 0.5,
) -> np.ndarray:
    """Compute IRIS CCD dark frame using Eq. 1 from De Pontieu et al. (2014).

    Args:
        t_ccd: CCD temperature in Celsius.
        t_ceb: CEB temperature in Celsius.
        t_int: Integration time in seconds.
        nx: Spectral on-chip summing factor (1, 2, 4, or 8).
        ny: Spatial on-chip summing factor (1, 2, or 4).
        npix_x: Number of pixels along wavelength direction.
        a: Dark current coefficient a_j.
        b: Dark current coefficient b_j (per degree C).
        pedestal_base: Base pedestal level in DN.
        pedestal_slope: Pedestal sensitivity to CEB temperature (DN per °C).

    Returns:
        1D array of dark level [DN] along the wavelength direction.
    """
    x = np.arange(npix_x)

    # Term 1: Pedestal (function of CEB temperature)
    pedestal = pedestal_base + pedestal_slope * (t_ceb - 20.0)

    # Term 2: Dark current (exponential in CCD temperature)
    dark_current = np.exp(a + b * t_ccd) * nx * ny * t_int

    # Term 3: Shape correction Delta_D (simplified linear ramp)
    # At t_int ~ 0: flat; increases to bilinear at longer t_int
    slope = 10.0 * t_int / 30.0  # ~10 DN end-to-end at 32x summing, 30s
    delta_d = slope * (x / npix_x - 0.5) * nx * ny

    dark_total = pedestal + dark_current + delta_d
    return dark_total


# --- Demonstrate dark model for different conditions ---
x = np.arange(2061)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Effect of CCD temperature
ax = axes[0]
for t_ccd in [-50, -40, -30, -20]:
    dark = iris_dark_model(t_ccd=t_ccd, t_ceb=22.0, t_int=8.0, nx=1, ny=1)
    ax.plot(x, dark, label=f'T_CCD = {t_ccd}°C')
ax.set_xlabel('Pixel (wavelength direction)')
ax.set_ylabel('Dark Level [DN]')
ax.set_title('(a) CCD Temperature Effect\n(a) CCD 온도 효과')
ax.legend(fontsize=9)

# (b) Effect of integration time
ax = axes[1]
for t_int in [0.5, 2, 8, 15, 30]:
    dark = iris_dark_model(t_ccd=-40.0, t_ceb=22.0, t_int=t_int, nx=1, ny=1)
    ax.plot(x, dark, label=f't_int = {t_int} s')
ax.set_xlabel('Pixel (wavelength direction)')
ax.set_ylabel('Dark Level [DN]')
ax.set_title('(b) Integration Time Effect\n(b) 적분 시간 효과')
ax.legend(fontsize=9)

# (c) Effect of on-chip summing
ax = axes[2]
for (nx, ny) in [(1, 1), (2, 2), (4, 4), (2, 8)]:
    dark = iris_dark_model(t_ccd=-40.0, t_ceb=22.0, t_int=8.0, nx=nx, ny=ny)
    ax.plot(x, dark, label=f'nx={nx}, ny={ny}')
ax.set_xlabel('Pixel (wavelength direction)')
ax.set_ylabel('Dark Level [DN]')
ax.set_title('(c) On-chip Summing Effect\n(c) 온칩 합산 효과')
ax.legend(fontsize=9)

fig.suptitle('IRIS CCD Dark Current Model (Eq. 1, De Pontieu et al. 2014)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2. IRIS Thermal Coverage / IRIS 열적 커버리지

Visualization of IRIS spectral lines and their formation temperatures (Table 4, p. 2748).
IRIS covers temperatures from log T = 3.7 (photosphere, ~5000 K) to log T = 7.0 (flare corona, ~10 MK).

Table 4(p. 2748)의 IRIS 분광선과 형성 온도를 시각화합니다.
IRIS는 log T = 3.7 (광구, ~5000 K)에서 log T = 7.0 (플레어 코로나, ~10 MK)까지 커버합니다.

In [ ]:
# IRIS spectral lines from Table 4 (p. 2748)
iris_lines = [
    # (Ion, wavelength [Å], log T [K], band, CEB)
    ('Mg II wing', 2820.0, 3.8, 'NUV', 2),
    ('O I',        1355.6, 3.8, 'FUV 1', 1),
    ('Mg II h',    2803.5, 4.0, 'NUV', 2),
    ('Mg II k',    2796.4, 4.0, 'NUV', 2),
    ('C II',       1334.5, 4.3, 'FUV 1', 1),
    ('C II',       1335.7, 4.3, 'FUV 1', 1),
    ('Si IV',      1402.8, 4.8, 'FUV 2', 1),
    ('Si IV',      1393.8, 4.8, 'FUV 2', 1),
    ('O IV',       1399.8, 5.2, 'FUV 2', 1),
    ('O IV',       1401.2, 5.2, 'FUV 2', 1),
    ('Fe XII',     1349.4, 6.2, 'FUV 1', 1),
    ('Fe XXI',     1354.1, 7.0, 'FUV 1', 1),
]

# Solar atmosphere regions
regions = [
    (3.5, 3.85, 'Photosphere\n광구', '#FFF3E0'),
    (3.85, 4.3, 'Chromosphere\n채층', '#FFE0B2'),
    (4.3, 5.5, 'Transition Region\n전이 영역', '#FFCC80'),
    (5.5, 6.5, 'Corona\n코로나', '#FFB74D'),
    (6.5, 7.2, 'Flare Corona\n플레어 코로나', '#FF8A65'),
]

fig, ax = plt.subplots(figsize=(14, 8))

# Draw atmosphere regions
for log_t_min, log_t_max, label, color in regions:
    ax.axhspan(log_t_min, log_t_max, alpha=0.3, color=color, zorder=0)
    ax.text(2860, (log_t_min + log_t_max) / 2, label,
            fontsize=9, va='center', ha='left', style='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# Color map for bands
band_colors = {'FUV 1': '#1565C0', 'FUV 2': '#E65100', 'NUV': '#2E7D32'}

# Plot spectral lines
for ion, wl, log_t, band, ceb in iris_lines:
    color = band_colors[band]
    ax.plot(wl, log_t, 'o', color=color, markersize=10, zorder=5,
            markeredgecolor='white', markeredgewidth=0.5)
    # Offset labels to avoid overlap
    offset_x = 8 if wl < 2000 else -60
    offset_y = 0.05
    # Handle C II and O IV duplicates
    if ion == 'C II' and wl == 1335.7:
        offset_y = -0.12
    if ion == 'O IV' and wl == 1401.2:
        offset_y = -0.12
    if ion == 'Si IV' and wl == 1393.8:
        offset_y = -0.12
    ax.annotate(f'{ion} {wl:.1f} Å',
                xy=(wl, log_t), xytext=(wl + offset_x, log_t + offset_y),
                fontsize=8, color=color, fontweight='bold',
                arrowprops=dict(arrowstyle='-', color=color, alpha=0.3))

# SG passband ranges
sg_bands = [
    (1331.7, 1358.4, 'FUV 1\n1332–1358 Å', '#1565C0'),
    (1389.0, 1407.0, 'FUV 2\n1389–1407 Å', '#E65100'),
    (2782.7, 2835.1, 'NUV\n2783–2835 Å', '#2E7D32'),
]
for wl_min, wl_max, label, color in sg_bands:
    ax.axvspan(wl_min, wl_max, alpha=0.08, color=color, zorder=0)
    ax.text((wl_min + wl_max) / 2, 3.55, label,
            fontsize=8, ha='center', va='bottom', color=color, fontweight='bold')

# Legend for bands
handles = [plt.Line2D([0], [0], marker='o', color=c, linestyle='None',
                       markersize=8, label=b)
           for b, c in band_colors.items()]
ax.legend(handles=handles, loc='upper left', fontsize=10)

ax.set_xlabel('Wavelength [Å]', fontsize=12)
ax.set_ylabel('log T [K]', fontsize=12)
ax.set_title('IRIS Thermal Coverage — Spectral Lines and Formation Temperatures\n'
             'IRIS 열적 커버리지 — 분광선과 형성 온도 (Table 4)',
             fontsize=13, fontweight='bold')
ax.set_ylim(3.4, 7.3)
ax.set_xlim(1310, 2900)

# Add temperature scale on right axis
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
temp_ticks = [3.7, 4.0, 4.3, 4.8, 5.2, 6.2, 7.0]
ax2.set_yticks(temp_ticks)
ax2.set_yticklabels([f'{10**t:.0e} K' for t in temp_ticks], fontsize=8)
ax2.set_ylabel('Temperature [K]', fontsize=12)

plt.tight_layout()
plt.show()

## 3. Raster Scan Simulation / 래스터 스캔 시뮬레이션

IRIS builds 2D spectral maps by scanning its slit across the Sun using PZT-driven secondary mirror (Section 5.3, p. 2755). We simulate the four basic raster modes from Table 12:

- **Dense raster**: step = slit width (0.33 arcsec) — complete spatial sampling
- **Sparse raster**: step = 1 arcsec — faster but undersampled
- **Coarse raster**: step = 2 arcsec — fastest scan of large areas
- **Sit-and-stare**: fixed slit position — temporal evolution only

IRIS는 PZT로 보조 거울을 구동하여 슬릿을 태양면 위로 스캔합니다 (Section 5.3, p. 2755).
Table 12의 4가지 기본 래스터 모드를 시뮬레이션합니다.

In [ ]:
def simulate_raster(
    fov_x: float = 30.0,
    fov_y: float = 60.0,
    slit_width: float = 0.33,
    step_size: float = 0.33,
    slit_length: float = 175.0,
    cadence_per_step: float = 3.0,
) -> dict:
    """Simulate an IRIS raster scan pattern.

    Args:
        fov_x: Raster width in arcsec (scan direction).
        fov_y: Slit length to display in arcsec.
        slit_width: Slit width in arcsec (0.33 for IRIS).
        step_size: Step between raster positions in arcsec.
        slit_length: Full slit length in arcsec.
        cadence_per_step: Time per raster step in seconds.

    Returns:
        Dictionary with raster positions, total time, and coverage info.
    """
    n_steps = int(fov_x / step_size)
    positions = np.arange(n_steps) * step_size
    total_time = n_steps * cadence_per_step
    coverage = slit_width / step_size  # fraction of area sampled
    return {
        'positions': positions,
        'n_steps': n_steps,
        'total_time': total_time,
        'coverage': min(coverage, 1.0),
        'step_size': step_size,
        'slit_width': slit_width,
        'fov_x': fov_x,
        'fov_y': fov_y,
    }


# Create synthetic solar "scene" (simplified chromospheric brightness)
np.random.seed(42)
scene_nx, scene_ny = 200, 400
scene_x = np.linspace(0, 60, scene_nx)  # arcsec
scene_y = np.linspace(0, 120, scene_ny)
XX, YY = np.meshgrid(scene_x, scene_y)

# Synthetic chromosphere: network + bright points + loop
scene = 50 + 20 * np.sin(2 * np.pi * XX / 15) * np.cos(2 * np.pi * YY / 20)
scene += 30 * np.exp(-((XX - 30)**2 + (YY - 60)**2) / 200)  # bright region
scene += np.random.normal(0, 5, scene.shape)  # noise

# --- Plot four raster modes ---
modes = [
    ('Dense (0.33")', 0.33, '#1565C0'),
    ('Sparse (1")', 1.0, '#E65100'),
    ('Coarse (2")', 2.0, '#2E7D32'),
    ('Sit-and-Stare', None, '#7B1FA2'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9),
                          gridspec_kw={'height_ratios': [3, 1]})

for i, (name, step, color) in enumerate(modes):
    ax_top = axes[0, i]
    ax_bot = axes[1, i]

    # Show scene
    ax_top.imshow(scene, extent=[0, 60, 0, 120], origin='lower',
                   cmap='hot', aspect='auto', alpha=0.4)

    if step is not None:
        raster = simulate_raster(fov_x=30.0, fov_y=60.0, step_size=step)
        # Draw slit positions
        for pos in raster['positions']:
            ax_top.axvline(pos + 15, color=color, alpha=0.5,
                          linewidth=max(0.33 / step * 2, 0.5))
        ax_top.set_title(f'{name}\nSteps: {raster["n_steps"]}, '
                        f'Time: {raster["total_time"]:.0f}s\n'
                        f'Coverage: {raster["coverage"]*100:.0f}%',
                        fontsize=10, color=color, fontweight='bold')

        # Bottom: time-position diagram
        times = np.arange(raster['n_steps']) * 3.0
        ax_bot.step(times, raster['positions'], color=color, linewidth=1.5)
        ax_bot.set_xlabel('Time [s]')
        ax_bot.set_ylabel('Slit X [arcsec]')
        ax_bot.set_ylim(0, 32)
    else:
        # Sit-and-stare: single slit
        ax_top.axvline(30, color=color, alpha=0.8, linewidth=3)
        ax_top.set_title(f'{name}\nFixed slit, continuous\n'
                        f'Best temporal resolution',
                        fontsize=10, color=color, fontweight='bold')
        times = np.arange(100) * 3.0
        ax_bot.axhline(15, color=color, linewidth=1.5)
        ax_bot.set_xlabel('Time [s]')
        ax_bot.set_ylabel('Slit X [arcsec]')
        ax_bot.set_ylim(0, 32)
        ax_bot.text(150, 18, 'Fixed position\n고정 위치',
                   fontsize=9, ha='center', color=color)

    ax_top.set_xlim(10, 50)
    ax_top.set_ylim(30, 90)
    ax_top.set_xlabel('Solar X [arcsec]')
    if i == 0:
        ax_top.set_ylabel('Solar Y [arcsec]')

fig.suptitle('IRIS Raster Scan Modes (Table 12, De Pontieu et al. 2014)\n'
             'IRIS 래스터 스캔 모드',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Rice Compression / Rice 압축

IRIS uses lossless Rice compression (Rice & Plaunt, 1971) as the primary data compression algorithm (Section 7.4, p. 2762). The algorithm encodes running differences between consecutive pixels, achieving ~2:1 compression for UV spectra and images.

IRIS는 무손실 Rice 압축을 주요 데이터 압축 알고리즘으로 사용합니다 (Section 7.4, p. 2762).
연속 픽셀 간의 차이를 인코딩하여 UV 스펙트라와 영상에서 ~2:1 압축비를 달성합니다.

### Algorithm / 알고리즘:
1. Compute running differences: $\delta_i = x_i - x_{i-1}$
2. Split each $\delta_i$ into: K least significant bits (stored directly) + quotient (stored as unary code)
3. The K-value controls the split point — optimized per observation type

In [ ]:
def rice_encode(data: np.ndarray, k: int = 4) -> dict:
    """Simplified Rice compression encoder.

    Encodes running differences using Rice coding: each difference is split
    into a quotient (unary coded) and remainder (k bits, binary coded).

    Args:
        data: 1D array of integer pixel values.
        k: Number of least significant bits stored directly (Rice K-value).

    Returns:
        Dictionary with encoded data and compression statistics.
    """
    # Step 1: Running differences (first value stored as-is)
    diffs = np.diff(data.astype(np.int64))

    # Map signed to unsigned (zig-zag encoding for Rice)
    mapped = np.where(diffs >= 0, 2 * diffs, -2 * diffs - 1).astype(np.uint64)

    # Step 2: Split into quotient and remainder
    m = 1 << k  # 2^k
    quotients = mapped >> k  # equivalent to mapped // m
    remainders = mapped & (m - 1)  # equivalent to mapped % m

    # Step 3: Count bits
    # Quotient bits: q+1 per value (q zeros + 1 stop bit)
    quotient_bits = np.sum(quotients + 1)
    # Remainder bits: k per value
    remainder_bits = len(remainders) * k
    # First value: stored as full 14-bit integer (IRIS nominal bit depth)
    first_value_bits = 14

    total_bits = int(first_value_bits + quotient_bits + remainder_bits)
    original_bits = len(data) * 14  # 14-bit data
    ratio = original_bits / total_bits

    return {
        'original_bits': original_bits,
        'compressed_bits': total_bits,
        'compression_ratio': ratio,
        'k_value': k,
        'n_pixels': len(data),
        'mean_diff': np.mean(np.abs(diffs)),
        'max_diff': np.max(np.abs(diffs)),
    }


def rice_decode(first_value: int, quotients: np.ndarray,
                remainders: np.ndarray, k: int) -> np.ndarray:
    """Rice decoder (conceptual — inverse of encoding).

    Args:
        first_value: First pixel value (stored uncompressed).
        quotients: Array of quotient values.
        remainders: Array of remainder values (k-bit).
        k: Rice K-value.

    Returns:
        Reconstructed pixel data array.
    """
    m = 1 << k
    mapped = quotients * m + remainders
    # Inverse zig-zag
    diffs = np.where(mapped % 2 == 0, mapped // 2, -(mapped + 1) // 2)
    data = np.empty(len(diffs) + 1, dtype=np.int64)
    data[0] = first_value
    data[1:] = first_value + np.cumsum(diffs)
    return data


# --- Generate synthetic IRIS-like data and test compression ---
np.random.seed(42)

# Simulate different types of IRIS data
def make_synthetic_spectrum(n=2061, base=100, line_strength=500):
    """Create synthetic UV spectrum with emission lines."""
    x = np.arange(n)
    # Continuum + noise
    spectrum = base + np.random.poisson(base, n).astype(float)
    # Add emission lines (like C II, Si IV)
    for center in [400, 450, 1200, 1300]:
        width = np.random.uniform(3, 8)
        spectrum += line_strength * np.exp(-(x - center)**2 / (2 * width**2))
    return np.clip(spectrum, 0, 16383).astype(np.int32)  # 14-bit range


def make_synthetic_sji(n=2061, base=200):
    """Create synthetic SJI image row with gradual variation."""
    x = np.arange(n)
    # Smooth background + structure
    image_row = base + 50 * np.sin(2 * np.pi * x / 500)
    image_row += np.random.poisson(np.maximum(image_row, 1)).astype(float)
    return np.clip(image_row, 0, 16383).astype(np.int32)


# Test different K values
spectrum = make_synthetic_spectrum()
sji_row = make_synthetic_sji()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Spectrum and its running differences
ax = axes[0, 0]
ax.plot(spectrum[:500], 'k-', linewidth=0.5, alpha=0.8)
ax.set_title('(a) Synthetic FUV Spectrum (first 500 pixels)\n합성 FUV 스펙트럼')
ax.set_xlabel('Pixel')
ax.set_ylabel('DN')

ax = axes[0, 1]
diffs = np.diff(spectrum[:500])
ax.plot(diffs, 'b-', linewidth=0.5, alpha=0.8)
ax.axhline(0, color='r', linestyle='--', alpha=0.5)
ax.set_title('(b) Running Differences $\\delta_i = x_i - x_{i-1}$\n연속 차분')
ax.set_xlabel('Pixel')
ax.set_ylabel('$\\Delta$ DN')

# (c) Compression ratio vs K-value
ax = axes[1, 0]
k_values = range(1, 11)
ratios_spectrum = [rice_encode(spectrum, k=k)['compression_ratio'] for k in k_values]
ratios_sji = [rice_encode(sji_row, k=k)['compression_ratio'] for k in k_values]

ax.plot(list(k_values), ratios_spectrum, 'bo-', label='FUV Spectrum', linewidth=2)
ax.plot(list(k_values), ratios_sji, 'rs-', label='SJI Image Row', linewidth=2)
ax.axhline(2.0, color='gray', linestyle='--', alpha=0.5, label='Target ~2:1')
ax.set_xlabel('Rice K-value')
ax.set_ylabel('Compression Ratio')
ax.set_title('(c) Compression Ratio vs K-value\n압축비 vs K값')
ax.legend()

# (d) Verify lossless: encode then conceptually decode
ax = axes[1, 1]
best_k_spec = list(k_values)[np.argmax(ratios_spectrum)]
result = rice_encode(spectrum, k=best_k_spec)
ax.bar(['Original\n원본', 'Compressed\n압축'],
       [result['original_bits'] / 8 / 1024, result['compressed_bits'] / 8 / 1024],
       color=['#1565C0', '#E65100'])
ax.set_ylabel('Size [KB]')
ax.set_title(f'(d) FUV Spectrum Compression (K={best_k_spec})\n'
             f'Ratio: {result["compression_ratio"]:.2f}:1')
ax.text(0, result['original_bits'] / 8 / 1024 + 0.1,
        f'{result["original_bits"] / 8 / 1024:.1f} KB', ha='center', fontsize=10)
ax.text(1, result['compressed_bits'] / 8 / 1024 + 0.1,
        f'{result["compressed_bits"] / 8 / 1024:.1f} KB', ha='center', fontsize=10)

fig.suptitle('Rice Compression for IRIS Data (Section 7.4)\n'
             'IRIS 데이터의 Rice 압축',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Best K-value for spectrum: {best_k_spec} "
      f"(ratio: {max(ratios_spectrum):.2f}:1)")
print(f"Best K-value for SJI: {list(k_values)[np.argmax(ratios_sji)]} "
      f"(ratio: {max(ratios_sji):.2f}:1)")
print(f"Mean |diff| for spectrum: {result['mean_diff']:.1f} DN")
print(f"Max |diff| for spectrum: {result['max_diff']} DN")

## 5. Orbital Wobble Model / 궤도 워블 모델

Thermal flexing between the guide telescope and the main IRIS telescope introduces a pointing wobble on orbital timescales (~97 minutes). The wobble is ~3 arcsec peak-to-peak in x and ~1 arcsec in y (Section 7.5, p. 2763).

Key finding: for roll angle $\alpha$, the wobble can be approximated by phase-shifting the 0° wobble curve:

$$\text{wobble}(\alpha, \phi) \approx \text{wobble}(0°, \phi + \alpha/360°)$$

GT와 IRIS 망원경 사이의 열변형으로 궤도 주기(~97분)의 지향 진동이 발생합니다.
x 방향 ~3 arcsec, y 방향 ~1 arcsec (peak-to-peak).
롤 각도 $\alpha$에서의 워블은 0° 워블 곡선을 $\alpha/360°$만큼 위상 이동하여 근사합니다.

In [ ]:
def orbital_wobble(
    phase: np.ndarray,
    roll_angle: float = 0.0,
    amp_x: float = 3.0,
    amp_y: float = 1.0,
    n_harmonics: int = 3,
) -> tuple[np.ndarray, np.ndarray]:
    """Model IRIS orbital wobble as a function of orbital phase.

    The wobble is modeled as a sum of harmonics of the orbital period,
    with phase shifted by roll_angle/360.

    Args:
        phase: Orbital phase array (0 to 1, where 0 = ascending node).
        roll_angle: Spacecraft roll angle in degrees.
        amp_x: Peak-to-peak amplitude in x direction [IRIS pixels].
        amp_y: Peak-to-peak amplitude in y direction [IRIS pixels].
        n_harmonics: Number of Fourier harmonics.

    Returns:
        Tuple of (wobble_x, wobble_y) in IRIS pixels.
    """
    # Phase shift from roll angle (Eq. from Section 7.5)
    phase_shift = roll_angle / 360.0
    shifted_phase = phase + phase_shift

    # Model wobble as sum of harmonics (simplified from actual calibration data)
    wobble_x = np.zeros_like(phase)
    wobble_y = np.zeros_like(phase)

    # Coefficients inspired by Figure 14 (approximate)
    for n in range(1, n_harmonics + 1):
        wobble_x += (amp_x / (2 * n)) * np.sin(2 * np.pi * n * shifted_phase + 0.3 * n)
        wobble_y += (amp_y / (2 * n)) * np.cos(2 * np.pi * n * shifted_phase + 0.5 * n)

    return wobble_x, wobble_y


# --- Reproduce Figure 14-like plot ---
phase = np.linspace(0, 1, 500)
roll_angles = [0, 90, -90]
colors = ['black', 'red', 'blue']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Wobble for different roll angles (like Figure 14 left panel)
ax = axes[0, 0]
for roll, color in zip(roll_angles, colors):
    wx, wy = orbital_wobble(phase, roll_angle=roll)
    ax.plot(phase, wx, color=color, linewidth=1.5,
            label=f'roll = {roll}°', linestyle='-')
    ax.plot(phase, wy, color=color, linewidth=1.5,
            linestyle='--', alpha=0.7)
ax.set_xlabel('Orbital Phase')
ax.set_ylabel('Wobble [IRIS pixels]')
ax.set_title('(a) Orbital Wobble (cf. Figure 14, left)\n궤도 워블')
ax.legend(fontsize=9)
ax.text(0.02, 0.95, 'Solid: x, Dashed: y', transform=ax.transAxes,
        fontsize=9, va='top')

# (b) Phase-shifted curves overlaid (like Figure 14 right panel)
ax = axes[0, 1]
wx_0, _ = orbital_wobble(phase, roll_angle=0)
wx_90, _ = orbital_wobble(phase, roll_angle=90)
wx_m90, _ = orbital_wobble(phase, roll_angle=-90)

# Shift +90 by -0.25 and -90 by +0.25 to overlay on 0°
ax.plot(phase, wx_0, 'k-', linewidth=2, label='roll = 0°')
ax.plot(phase, np.roll(wx_90, -int(0.25 * len(phase))),
        'r--', linewidth=1.5, label='roll = +90° (shifted −0.25)')
ax.plot(phase, np.roll(wx_m90, int(0.25 * len(phase))),
        'b:', linewidth=1.5, label='roll = −90° (shifted +0.25)')
ax.set_xlabel('Orbital Phase')
ax.set_ylabel('Wobble x [IRIS pixels]')
ax.set_title('(b) Phase-Shifted Overlay (cf. Figure 14, right)\n위상 이동 중첩')
ax.legend(fontsize=9)

# (c) Wobble trajectory (x vs y) over one orbit
ax = axes[1, 0]
for roll, color in zip(roll_angles, colors):
    wx, wy = orbital_wobble(phase, roll_angle=roll)
    ax.plot(wx, wy, color=color, linewidth=1.5, label=f'roll = {roll}°')
    ax.plot(wx[0], wy[0], 'o', color=color, markersize=8)  # start
ax.set_xlabel('Wobble X [IRIS pixels]')
ax.set_ylabel('Wobble Y [IRIS pixels]')
ax.set_title('(c) Wobble Trajectory over One Orbit\n한 궤도 동안의 워블 궤적')
ax.legend(fontsize=9)
ax.set_aspect('equal')

# (d) Wobble correction residual
ax = axes[1, 1]
wx_raw, wy_raw = orbital_wobble(phase, roll_angle=0)
# OWT correction (simplified: subtract best-fit single harmonic)
correction_x = wx_raw.max() / 2 * np.sin(2 * np.pi * phase + 0.3)
correction_y = wy_raw.max() / 2 * np.cos(2 * np.pi * phase + 0.5)
residual_x = wx_raw - correction_x
residual_y = wy_raw - correction_y

pixel_scale = 0.167  # arcsec per IRIS pixel
ax.plot(phase, residual_x * pixel_scale, 'b-', label='Residual x', linewidth=1.5)
ax.plot(phase, residual_y * pixel_scale, 'r--', label='Residual y', linewidth=1.5)
ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax.axhspan(-0.167 * 2, 0.167 * 2, alpha=0.1, color='green',
           label='±2 pixel threshold')
ax.set_xlabel('Orbital Phase')
ax.set_ylabel('Residual [arcsec]')
ax.set_title('(d) Wobble Correction Residual\n워블 보정 잔차')
ax.legend(fontsize=9)

fig.suptitle('IRIS Orbital Wobble Model (Section 7.5, De Pontieu et al. 2014)\n'
             'IRIS 궤도 워블 모델',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print wobble statistics
print(f"Wobble peak-to-peak: x = {wx_raw.max() - wx_raw.min():.1f} pixels "
      f"({(wx_raw.max() - wx_raw.min()) * pixel_scale:.2f} arcsec)")
print(f"Wobble peak-to-peak: y = {wy_raw.max() - wy_raw.min():.1f} pixels "
      f"({(wy_raw.max() - wy_raw.min()) * pixel_scale:.2f} arcsec)")
print(f"Residual RMS after correction: x = {np.std(residual_x) * pixel_scale:.3f} arcsec")
print(f"Residual RMS after correction: y = {np.std(residual_y) * pixel_scale:.3f} arcsec")
print(f"Orbital period: ~97 minutes")

## Summary / 요약

This notebook implemented five key algorithms and concepts from the IRIS instrument paper:

| Algorithm / 알고리즘 | Section | Key Result / 핵심 결과 |
|---|---|---|
| CCD Dark Current Model | 7.1 (Eq. 1) | Dark level depends exponentially on CCD temperature; on-chip summing amplifies both dark current and shape variation / 다크 레벨은 CCD 온도에 지수적 의존; 온칩 합산이 다크 전류와 형상 변화를 증폭 |
| Thermal Coverage | Table 4 | IRIS covers log T = 3.7–7.0 (5000 K to 10 MK) with 12 spectral lines across 3 bands / 3개 대역의 12개 분광선으로 log T = 3.7–7.0 커버 |
| Raster Scan Modes | Table 12 | Dense rasters provide full coverage but take ~270s; sit-and-stare gives best temporal resolution / 밀집 래스터는 완전 커버리지(~270초); sit-and-stare는 최고 시간 해상도 |
| Rice Compression | 7.4 | Achieves ~2:1 lossless compression by encoding running differences; optimal K depends on data smoothness / 연속 차분 인코딩으로 ~2:1 무손실 압축; 최적 K는 데이터 평활도에 의존 |
| Orbital Wobble | 7.5 | ~3 arcsec wobble in x per orbit; phase-shift approximation enables correction at all roll angles / 궤도당 x 방향 ~3 arcsec 워블; 위상 이동 근사로 모든 롤 각도에서 보정 가능 |